# Diffusion 1d

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget
from src.Diffusion1d import *
from src.trainloop import *
import numpy as np
import pandas as pd
import joblib
from src.DataConfig import DataConfig
from src.ConditioningDropout import CondDropout
import tempfile, torch
from src.DataPipeline import DataPipeline, make_splits, add_share_cols, hash_checker
from ema_pytorch import EMA
import wandb
wandb.login()
device = "cuda" if torch.cuda.is_available() else "cpu"

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Samuel\_netrc.
wandb: Currently logged in as: natusvictus (natusvictus-fh-aachen-university-of-applied-sciences) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Load Data and set Config Params

In [28]:
# Data Locations
target_root = Path(r"../data/split")

split_dirs = {
    "train": target_root/"train",
    "val": target_root/"val",
    "test": target_root/"test",
    "holdout_building": target_root/"building_holdout",
    "holdout_year": target_root/"year_holdout",
    "holdout_station": target_root/"station_holdout",

}
MANIFEST_DIR = Path(r"../data/manifests")
manifests = load_split_manifests({
    "train": MANIFEST_DIR/"manifest_train.parquet",
    "val":   MANIFEST_DIR/"manifest_val.parquet",
    "test":  MANIFEST_DIR/"manifest_test.parquet",
    "holdout_building": MANIFEST_DIR/"manifest_holdout_building.parquet",
    "holdout_year": MANIFEST_DIR/"manifest_holdout_year.parquet",
    "holdout_station": MANIFEST_DIR/"manifest_holdout_station.parquet",
})

train_df = manifests["train"]
strat_cols = ["buildingType", "buildingAgeBin"]
# all dynamic columns for stat caclculation
STATS_COLS =  ["Electricity Load", "Hot Water Load","Heating Load","Occupancy Home Active", "Occupancy Home Not Active","T","Off Day"]
# Aggregation Method in case of other than 1min Res
TS_AGG = {
    "T": "first",
    "Occupancy Home Active": "first",
    "Occupancy Home Not Active": "first",
    "Electricity Load": "mean",
    "Hot Water Load": "mean",
    "Heating Load": "mean",
    "Off Day": "max"
}

train: 11700 rows, 9750 buildings
val: 2340 rows, 390 buildings
test: 1755 rows, 195 buildings
holdout_building: 60 rows, 10 buildings
holdout_year: 351 rows, 117 buildings
holdout_station: 702 rows, 117 buildings


In [29]:
# todo check what happens when no conditioning is used
# Cols to Train on and Generate
TARGET_COLS = ["Electricity Load", "Hot Water Load","Heating Load"] #"T", "Occupancy Home Not Active", "Occupancy Home Not Active","Heating Load", "Electricity Load", "Hot Water Load"

# Timeseries Cols to Condition on
DYNAMIC_COND_COLS = ["T", "Occupancy Home Active", "Occupancy Home Not Active","Off Day"] #"T", "Occupancy Home Active", "Occupancy Home Not Active", "T"

# Context Use i.e. steps of previous window as conditioning
CONTEXT_STEPS = 60
CONTEXT_REVERSE = True
N_CTX = len(TARGET_COLS) + 1 if CONTEXT_STEPS > 0 else 0

# doy as conditioning
USE_SEASON = True
N_SEASON = 2 if USE_SEASON else 0
N_DY_CON_COLS = len(DYNAMIC_COND_COLS) + N_SEASON + N_CTX

# Static Parameters to condition on
STATIC_CAT_COLS = ["buildingType","buildingAgeBin","year_type","climateRegion"]
#"buildingType","buildingAgeBin","year_type","climateRegion"
STATIC_BOOL_COLS = ["future"] #"future"
STATIC_CONT_COLS = ["a_ref","n_apartments","n_persons_sum"] + add_share_cols(train_df)
#"a_ref","n_apartments","n_persons_sum


# Cols for normalization of the Target cols so that the load distribution isnt weird across buildings
# define additional normalization by area/occpancy per domain
NORM_METHOD="none"

norm_variant={
    "none": {},
    "elec_sh_scale": {"Electricity Load": "n_persons_sum","Heating Load": "a_ref"},
    "elec_dhw_sh_scale": {"Electricity Load": "n_persons_sum","Hot Water Load": "n_persons_sum","Heating Load": "a_ref"}
}
NORM_BY_COL = norm_variant[NORM_METHOD]



#Cols where mean is set to zero for scaling
ZERO_COLS = []

# define indivdual scalers per column default is zscore
SCALER_BY_COL = {
    "Electricity Load": "log_zscore",
    "Hot Water Load": "log_zscore",
    "Heating Load": "log_zscore",
    "Off Day": "minmax"
}

# dropout groups and probs
cond_groups = {
    # "occupancy": ["Occupancy Home Active","Occupancy Home Not Active"],
    # "temperature": ["T"],
    # "off_day": ["Off Day"],
    # "season": "season"
}

cond_dropout = {
    # "occupancy": 0.2,
    # "temperature": 0.3,
    # "off_day": 0.1,
    # "season": 0.1
}

static_dropout={
    # "cat": 0.1, "cont": 0.1
}


cond_dropout_all=0.1

In [30]:
# Parameters for Windowing and Dataloader
WINDOW_DAYS = 1
STRIDE_DAYS = WINDOW_DAYS
RESOLUTION = "1min"

STEPS_PER_DAY = stepsperday(RESOLUTION)
WINDOW_LENGTH = WINDOW_DAYS * STEPS_PER_DAY
STRIDE = STRIDE_DAYS * STEPS_PER_DAY

In [31]:
cfg = DataConfig(
    target_cols=TARGET_COLS,
    dynamic_cond_cols=DYNAMIC_COND_COLS,
    static_cat_cols=STATIC_CAT_COLS,
    static_cont_cols=STATIC_CONT_COLS,
    static_bool_cols=STATIC_BOOL_COLS,
    resolution=RESOLUTION,
    window_days=WINDOW_DAYS,
    stride_days=STRIDE_DAYS,
    ts_agg=TS_AGG,
    stats_cols=STATS_COLS,
    norm_method=NORM_METHOD,
    norm_variant=norm_variant,
    scaler_by_col=SCALER_BY_COL,
    context_steps=CONTEXT_STEPS,
    context_reverse=CONTEXT_REVERSE,
    use_season=USE_SEASON,
    strat_cols=strat_cols,
    split_dirs={k: str(v) for k, v in split_dirs.items()},
    cond_groups=cond_groups,
    cond_dropout=cond_dropout,
    cond_dropout_all=cond_dropout_all,
    static_dropout=static_dropout,
    buffer_size=50,
)

In [32]:
cond = CondDropout.from_config(cfg)
pipe = DataPipeline.build(cfg, manifests)
schedule = DiffusionSchedule.make("sigmoid", timesteps=1000).to(device)

preprocessor = pipe.preprocessor
cat_cardinalities = pipe.cat_cardinalities
stats = pipe.stats
target_scaler = pipe.target_scaler
condition_scaler= pipe.condition_scaler
X0_MIN1, X0_MAX1 = pipe.x0_minmax()

train: 11700 rows, 9750 buildings
val: 2340 rows, 390 buildings
test: 1755 rows, 195 buildings
holdout_building: 60 rows, 10 buildings
holdout_year: 351 rows, 117 buildings
holdout_station: 702 rows, 117 buildings
Missing Files: 10200
Existing Files: 1500
Total: 11700
Missing Files: 2090
Existing Files: 250
Total: 2340
Missing Files: 1505
Existing Files: 250
Total: 1755
Missing Files: 0
Existing Files: 60
Total: 60
Missing Files: 201
Existing Files: 150
Total: 351
Missing Files: 552
Existing Files: 150
Total: 702


In [42]:
preprocessor.static_cat_cols

['buildingType', 'buildingAgeBin', 'year_type', 'climateRegion']

In [44]:

train_df_exist, val_df_exist, test_df_exist = (pipe.split_dfs[s] for s in ("train", "val", "test"))
train_files, train_run_ids, train_static = (pipe.prepare_data("train")[k] for k in ("files", "run_ids", "static"))
val_files, val_run_ids, val_static = (pipe.prepare_data("val")[k] for k in ("files", "run_ids", "static"))
test_files, test_run_ids, test_static = (pipe.prepare_data("test")[k] for k in ("files", "run_ids", "static"))

train_dataset = pipe.dataset("train", train=True)

train_dataset_eval = pipe.windowdataset("train", pipe.Window_drawer("train", 1, 1, seed=0))
val_dataset_eval = pipe.windowdataset("val", pipe.Window_drawer("val", 1, 1, seed=0))

In [46]:
BATCH_SIZE = 32
train_dataloader = pipe.dataloader(train_dataset, BATCH_SIZE, train=True)

train_eval_dataloader = pipe.dataloader(train_dataset_eval, BATCH_SIZE)
val_eval_dataloader = pipe.dataloader(val_dataset_eval, BATCH_SIZE)

In [47]:
ds = val_eval_dataloader.dataset
print("windows in index:", len(ds))
print("runs/files:", len(ds.files))
print("batch_size:", val_eval_dataloader.batch_size,
      "| drop_last:", val_eval_dataloader.drop_last,
      "| num_workers:", val_eval_dataloader.num_workers)
print("batches yielded:", sum(1 for _ in val_eval_dataloader))

windows in index: 468
runs/files: 39
batch_size: 32 | drop_last: False | num_workers: 4
batches yielded: 16


## Training Loop

In [51]:
EPOCHS = 10
LOSS_TYPE =  "l2"

In [52]:
model_cfg = make_model_cfg(pipe=pipe,
                           model_type="1d",
                           dim=64,
                           dim_mults=(1,2,4),
                           mid_attn="full",
                           level_attn="none",
                           use_day_index=False)

optim_cfg = dict(lr=1e-4)
ema_cfg = dict(beta=0.999, update_after_step=100, update_every=3, power=2/3)

model = build_model(model_cfg, device)
optimizer = build_optimizer(model, optim_cfg)
ema = build_ema(model, ema_cfg)
BOUNDARY_WEIGHT = [4.0, 1.0, 10.0]
config = {
    "epochs": EPOCHS, "timesteps": schedule.timesteps, "schedule": "sigmoid",
    "loss_type": LOSS_TYPE, "lr": optimizer.param_groups[0]["lr"],
    "model": model_cfg, "optimizer": optim_cfg, "ema": ema_cfg,
    "data": cfg.to_dict(), "split_hash": pipe.split_hash, "boundary_weight": BOUNDARY_WEIGHT
}
X0_MIN1, X0_MAX1 = pipe.x0_minmax()
extras = {"x0_min": X0_MIN1, "x0_max": X0_MAX1}
exp_trgt_cols = ''.join(s.lower().replace('load', '').replace(' ', '') for s in cfg.target_cols)
experiment = f"{model_cfg["model_type"]}_{exp_trgt_cols}"
run_dir = make_run_dir(config, root=r"../outputs/runs", experiment=experiment)
pipe.save_files(run_dir / "data")

saved data pipeline (split b679f6a63d73) to ..\outputs\runs\20261115_122718_29b638_1d_electricityhotwaterheating\data


In [53]:
comment = "Rework_Test"
run = wandb.init(project="diffusion-model_test_runs", name=f"{comment}_{run_dir.name}",group="1d_run_tests", config=config)
train(model,
      schedule,
      train_dataloader,
      optimizer,
      ema,
      device,
      epochs=EPOCHS,
      loss_type=LOSS_TYPE,
      run=run,
      run_dir=run_dir,
      config=config,
      extras=extras,
      val_dataloader=val_eval_dataloader,
      train_eval_dataloader=train_eval_dataloader,
      val_every_steps=5000,
      val_batches=None,
      ckpt_every_steps=5000,
      val_raw=False,
      boundary_weight=BOUNDARY_WEIGHT,
      cond=cond)
run.finish()

epoch 1/10: 0it [00:00, ?it/s]

Loss: 1.2637
Loss: 0.2475
Loss: 0.1450
Loss: 0.0433
Loss: 0.0717
Loss: 0.1175
Loss: 0.0459
Loss: 0.0723
Loss: 0.0421
Loss: 0.0897
Loss: 0.0740
Loss: 0.0386
Loss: 0.1160
Loss: 0.0338
Loss: 0.0452
Loss: 0.0405
Loss: 0.0355
Loss: 0.1028
Loss: 0.0193
Loss: 0.0338
Loss: 0.0525
Loss: 0.0297
Loss: 0.0632
Loss: 0.1021
Loss: 0.0856
Loss: 0.0551
Loss: 0.0356
Loss: 0.0443
Loss: 0.0777
Loss: 0.1040
Loss: 0.0477
Loss: 0.0528
Loss: 0.0297
Loss: 0.0295
Loss: 0.0789
Loss: 0.0902
Loss: 0.0375
Loss: 0.0729
Loss: 0.0742
Loss: 0.0753
Loss: 0.0520
Loss: 0.0693
Loss: 0.0496
Loss: 0.0307
Loss: 0.0407
Loss: 0.0385
Loss: 0.0276
Loss: 0.0603
Loss: 0.0315
Loss: 0.0447
step 5000: val_loss_ema=0.0701
Loss: 0.0370
Loss: 0.0914
Loss: 0.0142
Loss: 0.0148
Loss: 0.0426
Loss: 0.0210
Loss: 0.0688
Loss: 0.0232
Loss: 0.0512
Loss: 0.0792
Loss: 0.0383
Loss: 0.0263
Loss: 0.0276
Loss: 0.0250
Loss: 0.0360
Loss: 0.0261
Loss: 0.0160
Loss: 0.0401
Loss: 0.0566
Loss: 0.0559
Loss: 0.0426
Loss: 0.0624
Loss: 0.0253
Loss: 0.0218
Loss: 0

epoch 2/10: 0it [00:00, ?it/s]

Loss: 0.0248
Loss: 0.0159
Loss: 0.0182
Loss: 0.0453
Loss: 0.0290
Loss: 0.0383
Loss: 0.0250
Loss: 0.0223
Loss: 0.0670
Loss: 0.0705
Loss: 0.0428
Loss: 0.0251
Loss: 0.0206
Loss: 0.0625
Loss: 0.0363
Loss: 0.0181
Loss: 0.0673
Loss: 0.0262
Loss: 0.0204
Loss: 0.0147
Loss: 0.0546
Loss: 0.0447
Loss: 0.0484
Loss: 0.0282
Loss: 0.0112
Loss: 0.0336
Loss: 0.0175
Loss: 0.0732
Loss: 0.0144
step 20000: val_loss_ema=0.0486
Loss: 0.0225
Loss: 0.0130
Loss: 0.0429
Loss: 0.0484
Loss: 0.0485
Loss: 0.0237
Loss: 0.0567
Loss: 0.0509
Loss: 0.0449
Loss: 0.0858
Loss: 0.0543
Loss: 0.0184
Loss: 0.0148
Loss: 0.0269
Loss: 0.0182
Loss: 0.0176
Loss: 0.0452
Loss: 0.0204
Loss: 0.0108
Loss: 0.0320
Loss: 0.0253
Loss: 0.0644
Loss: 0.0241
Loss: 0.0214
Loss: 0.0187
Loss: 0.0156
Loss: 0.0758
Loss: 0.0131
Loss: 0.0483
Loss: 0.0290
Loss: 0.0282
Loss: 0.0230
Loss: 0.0723
Loss: 0.0492
Loss: 0.0525
Loss: 0.0286
Loss: 0.0169
Loss: 0.0265
Loss: 0.0307
Loss: 0.0400
Loss: 0.0744
Loss: 0.0255
Loss: 0.0332
Loss: 0.0284
Loss: 0.0187
Loss: 

epoch 3/10: 0it [00:00, ?it/s]

Loss: 0.0203
Loss: 0.0187
Loss: 0.0233
Loss: 0.0131
Loss: 0.0158
Loss: 0.0624
Loss: 0.0546
Loss: 0.0150
step 35000: val_loss_ema=0.0451
Loss: 0.0262
Loss: 0.0266
Loss: 0.0444
Loss: 0.0150
Loss: 0.0491
Loss: 0.0207
Loss: 0.0225
Loss: 0.0351
Loss: 0.0559
Loss: 0.0196
Loss: 0.0108
Loss: 0.0216
Loss: 0.0481
Loss: 0.0123
Loss: 0.0265
Loss: 0.0263
Loss: 0.0979
Loss: 0.0529
Loss: 0.0401
Loss: 0.0390
Loss: 0.0195
Loss: 0.0169
Loss: 0.0240
Loss: 0.0543
Loss: 0.0192
Loss: 0.0318
Loss: 0.0259
Loss: 0.0150
Loss: 0.0578
Loss: 0.0631
Loss: 0.0429
Loss: 0.0470
Loss: 0.0184
Loss: 0.0133
Loss: 0.0312
Loss: 0.0240
Loss: 0.0162
Loss: 0.0241
Loss: 0.0495
Loss: 0.0161
Loss: 0.0224
Loss: 0.0340
Loss: 0.0478
Loss: 0.0718
Loss: 0.0244
Loss: 0.0969
Loss: 0.0468
Loss: 0.0553
Loss: 0.0119
Loss: 0.0263
step 40000: val_loss_ema=0.0445
Loss: 0.0392
Loss: 0.0448
Loss: 0.0201
Loss: 0.0193
Loss: 0.0193
Loss: 0.0272
Loss: 0.0636
Loss: 0.0364
Loss: 0.0149
Loss: 0.0175
Loss: 0.0546
Loss: 0.0183
Loss: 0.0414
Loss: 0.0163


... (11 outputs left out)

Loss: 0.0669
Loss: 0.0220
Loss: 0.0328
Loss: 0.0199
Loss: 0.0262
Loss: 0.0192
Loss: 0.0553
Loss: 0.0198
Loss: 0.0560
Loss: 0.0284
Loss: 0.0307
Loss: 0.0305
Loss: 0.0158
Loss: 0.0165
Loss: 0.0183
Loss: 0.0309
Loss: 0.0147
Loss: 0.0130
Loss: 0.0728
Loss: 0.0237
Loss: 0.0545
Loss: 0.0543
Loss: 0.0166
Loss: 0.0151
Loss: 0.0338
Loss: 0.0666
Loss: 0.0315
Loss: 0.0203
Loss: 0.0172
Loss: 0.0285
Loss: 0.0124
Loss: 0.0185
step 140000: val_loss_ema=0.0411
Loss: 0.0146
Loss: 0.0500
Loss: 0.0181
Loss: 0.0729
Loss: 0.0346
Loss: 0.0196
Loss: 0.0208
Loss: 0.0132
Loss: 0.0174
Loss: 0.0472
Loss: 0.0125
Loss: 0.0101
Loss: 0.0146
Loss: 0.0720
Loss: 0.0192
Loss: 0.0238
Loss: 0.0405
Loss: 0.0545
Loss: 0.0262
Loss: 0.0484
Loss: 0.0397
Loss: 0.0228
Loss: 0.0276
Loss: 0.0184
Loss: 0.0202
Loss: 0.0309
Loss: 0.0370
Loss: 0.0450
Loss: 0.0628
Loss: 0.0148
Loss: 0.0285
Loss: 0.0464
Loss: 0.0762
Loss: 0.0197
Loss: 0.0412
Loss: 0.0480
Loss: 0.0193
Loss: 0.0269
Loss: 0.0528
Loss: 0.0379
Loss: 0.0112
Loss: 0.0573
Loss:

epoch 10/10: 0it [00:00, ?it/s]

Loss: 0.0247
Loss: 0.0455
Loss: 0.0143
Loss: 0.0578
Loss: 0.0181
Loss: 0.0690
Loss: 0.0347
Loss: 0.0112
Loss: 0.0094
Loss: 0.0592
Loss: 0.0282
step 155000: val_loss_ema=0.0410
Loss: 0.0224
Loss: 0.0105
Loss: 0.0144
Loss: 0.0142
Loss: 0.0431
Loss: 0.0516
Loss: 0.0516
Loss: 0.0640
Loss: 0.0646
Loss: 0.0179
Loss: 0.0536
Loss: 0.0310
Loss: 0.0562
Loss: 0.0142
Loss: 0.0139
Loss: 0.0498
Loss: 0.0567
Loss: 0.0125
Loss: 0.0281
Loss: 0.0162
Loss: 0.0213
Loss: 0.0273
Loss: 0.0159
Loss: 0.0452
Loss: 0.0183
Loss: 0.0288
Loss: 0.0170
Loss: 0.0185
Loss: 0.0188
Loss: 0.0129
Loss: 0.0395
Loss: 0.0261
Loss: 0.0324
Loss: 0.0231
Loss: 0.0725
Loss: 0.0205
Loss: 0.0172
Loss: 0.0135
Loss: 0.0107
Loss: 0.0168
Loss: 0.0169
Loss: 0.0383
Loss: 0.0186
Loss: 0.0164
Loss: 0.0237
Loss: 0.0248
Loss: 0.0263
Loss: 0.0511
Loss: 0.0167
Loss: 0.0177
step 160000: val_loss_ema=0.0409
Loss: 0.0531
Loss: 0.0252
Loss: 0.0187
Loss: 0.0246
Loss: 0.0516
Loss: 0.0344
Loss: 0.0194
Loss: 0.0531
Loss: 0.0508
Loss: 0.0141
Loss: 0.055

epoch,▁▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▃▃▃▃▄▄▄▄▆▆▆▆▆▆▆▇▇▇█████
epoch/loss,█▃▂▂▁▁▁▁▁▁
global_step,▁▁▁▁▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇███
train/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss,▇▂▂▂▃▆▂▂▅▄▂▂▁▆▂▁▃▂▂▇▅▂▂▂▅▂▂▆▂▁▅▅▁▆█▅▂▄▆▇
train_eval/loss_ema,█▅▄▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_eval/loss_ema_t0,█▅▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_eval/loss_ema_t111,█▅▄▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_eval/loss_ema_t222,█▅▄▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_eval/loss_ema_t333,█▅▄▄▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+17,...


In [20]:
model.ups

ModuleList(
  (0): ModuleList(
    (0): Sequential(
      (0): Upsample(scale_factor=2.0, mode='nearest')
      (1): Conv1d(256, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    )
    (1): ResnetBlock1d(
      (mlp): Sequential(
        (0): SiLU()
        (1): Linear(in_features=256, out_features=256, bias=True)
      )
      (block1): Block1d(
        (proj): WeightStandardizedConv1d(256, 128, kernel_size=(3,), stride=(1,), padding=(1,))
        (norm): GroupNorm(8, 128, eps=1e-05, affine=True)
        (act): SiLU()
      )
      (block2): Block1d(
        (proj): WeightStandardizedConv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
        (norm): GroupNorm(8, 128, eps=1e-05, affine=True)
        (act): SiLU()
      )
      (res_conv): Conv1d(256, 128, kernel_size=(1,), stride=(1,))
    )
  )
  (1): ModuleList(
    (0): Sequential(
      (0): Upsample(scale_factor=2.0, mode='nearest')
      (1): Conv1d(128, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    )
    (1